# Set 15 – Eigener Python-Code gehört zum Modell
Wir ergänzen zwei einfache Flächenmerkmale. Die Klasse steht in `custom_transformer.py`, damit sie auch in einem neuen Prozess unter demselben Modulnamen importiert werden kann. Eine ausschließlich in einer Notebook-Zelle definierte Klasse würde beim Laden häufig von der Notebook-Sitzung abhängen.

In [1]:
from pathlib import Path
import sys
kandidaten=[Path.cwd(),*Path.cwd().parents]
set_ordner=next(p for basis in kandidaten for p in (basis,basis/'set15-modelle-speichern-und-laden') if (p/'deployment_tools.py').exists())
if str(set_ordner) not in sys.path:sys.path.insert(0,str(set_ordner))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from deployment_tools import ARTEFAKTE,umgebung,json_speichern,eingaben_pruefen,neuer_prozess

In [2]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import joblib
iris=load_iris(as_frame=True)
X_train,X_test,y_train,y_test=train_test_split(iris.data,iris.target,test_size=0.25,random_state=42,stratify=iris.target)

In [3]:
from custom_transformer import FlaechenMerkmale
pipeline=Pipeline([('flaechen',FlaechenMerkmale()),('scaler',StandardScaler()),('modell',LogisticRegression(max_iter=300))])
pipeline.fit(X_train,y_train)
print('Test-Accuracy:',pipeline.score(X_test,y_test))
display(pipeline.named_steps['flaechen'].transform(X_test.head()))

Test-Accuracy: 0.9210526315789473


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),kelch_flaeche,blueten_flaeche
42,4.4,3.2,1.3,0.2,14.08,0.26
56,6.3,3.3,4.7,1.6,20.79,7.52
99,5.7,2.8,4.1,1.3,15.96,5.33
53,5.5,2.3,4.0,1.3,12.65,5.20
38,4.4,3.0,1.3,0.2,13.20,0.26


## Modellpaket speichern
Das Joblib-Artefakt enthält gelernte Zustände und den Verweis auf die Klasse. Es enthält nicht automatisch den Quellcode. In eine Zielumgebung gehören deshalb auch `custom_transformer.py`, die Anwendung und die dokumentierten Abhängigkeiten.

In [4]:
ordner=ARTEFAKTE/'sklearn_custom';ordner.mkdir(exist_ok=True)
joblib.dump(pipeline,ordner/'pipeline.joblib')
json_speichern(ordner/'config.json',{'feature_namen':list(X_train.columns),'klassen':iris.target_names.tolist(),'eigener_code':'custom_transformer.py'})
json_speichern(ordner/'versionen.json',umgebung(['scikit-learn','numpy','pandas','joblib']))
import shutil
shutil.copy2(set_ordner/'custom_transformer.py',ordner/'custom_transformer.py')
print('Artefakt mit zugehörigem Modul:',ordner.name)

Artefakt mit zugehörigem Modul: sklearn_custom


## Laden ohne die gefittete Pipeline im Speicher
Der separate Inference-Prozess importiert die Klasse aus dem Kursmodul. Der Quellcode wird zusätzlich im Artefaktordner dokumentiert. Zum Weitergeben wird das vollständige Set mit `predict.py` und seinen Hilfsmodulen verwendet; die Joblib-Datei allein genügt nicht.

In [5]:
probe=X_test.iloc[:6]
vorher=pipeline.predict_proba(probe)
del pipeline
extern=neuer_prozess('sklearn',ordner,probe.to_dict(orient='records'))
np.testing.assert_allclose(extern['wahrscheinlichkeiten'],vorher,rtol=1e-12,atol=1e-12)
print('Custom-Transformer wurde im neuen Prozess erfolgreich geladen.')
display(pd.DataFrame(extern['wahrscheinlichkeiten'],columns=iris.target_names))

Neuer Python-Prozess: Modell geladen; Vorhersagen ohne Training: 6
Custom-Transformer wurde im neuen Prozess erfolgreich geladen.


,setosa,versicolor,virginica
0,0.985694,0.014305,4.761693e-07
1,0.024957,0.701029,2.740137e-01
2,0.037644,0.914360,4.799652e-02
3,0.009948,0.933081,5.697062e-02
4,0.971611,0.028388,1.042601e-06
5,0.069781,0.727770,2.024495e-01


**Mini-Aufgabe zum Nachdenken:** Was würde fehlen, wenn nur `pipeline.joblib` auf einen anderen Rechner kopiert wird? Warum reicht eine Funktionsdefinition in einer alten Notebook-Zelle nicht?

Quelle: [scikit-learn: Model persistence](https://scikit-learn.org/stable/model_persistence.html). PDF-Bezug: Seiten 7–8 und 13.